# VGGFace2 Parent와 zero-shot 중간 평가

VGGFace2 공식 정렬 이미지에서 새 MobileFaceNet을 학습합니다. Epoch 2·5·10·18·24마다 LFW, SCFace, TinyFace를 평가합니다. [실행 순서와 결과 해석](HOW_TO_READ.md)

In [ ]:
import json
import os
import sys
from pathlib import Path

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "fine-tune/modules").is_dir())
sys.path.insert(0, str(repo / "fine-tune"))
from modules.clean_training import final_artifact, final_evaluation, recipe, run_directory, run_stage
from modules.clean_benchmarks import checked

DATA_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets"))
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models"))
RUN_ID = "carbon_mfn1024_s8972_v1"
EXECUTE = False
KEEP_RAW_RESULTS = False
SEED = 8972
RESUME = False
run = run_directory(MODEL_ROOT, RUN_ID)
stage = "parent"
print(run / stage)
print(json.dumps(recipe(stage), indent=2))

첫 코드 셀에서 실제 실행 여부와 원본 보존 여부를 지정합니다. 같은 실험의 세 노트북은 같은 RUN_ID를 사용합니다.

In [ ]:
if EXECUTE:
    result = run_stage(DATA_ROOT, run, stage, seed=SEED, keep_raw_results=KEEP_RAW_RESULTS, resume=RESUME)

In [ ]:
folder = run / stage
if (folder / "COMPLETED").exists():
    checkpoint, report = final_artifact(folder)
    print("완료", checkpoint, report["sha256"])
elif (folder / "progress.json").exists():
    report = json.loads((folder / "progress.json").read_bytes())
    print("부분 실행", report["epoch"], "epoch")
else:
    report = {}
    print("학습 시작 전")
for row in report.get("epochs", []):
    if "zero_shot" in row:
        m = row["zero_shot"]["metrics"]
        print(f'ep {row["epoch"]:2}  LFW {100*m["lfw"]["accuracy"]:.2f}%  '
              f'SCFace {100*m["scface"]["rank1"]:.2f}%  TinyFace {100*m["tinyface"]["rank1"]:.2f}%  '
              f'({row["zero_shot"]["seconds"]:.1f}s)', row["reference"])